In [2]:
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd
import os
import joblib
from sklearn.preprocessing import RobustScaler
from sklearn.mixture import BayesianGaussianMixture
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import classification_report, average_precision_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb

In [3]:
data = pd.read_csv(r'C:\Users\Anton\Downloads\archive (1)\creditcard.csv')
data = data.drop(['Time'], axis=1)

y = data['Class']
X = data.drop(['Class'], axis=1)

# scaling the amount to the other features
X['Amount'] = np.log(X['Amount'] + 0.001)
scaler = RobustScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])

data


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,4.356170,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,-0.975926,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,-0.484782,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,-0.399126,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


In [4]:
model_path = 'bgm_model_antoni.joblib'

if os.path.exists(model_path):
    print("Loading saved model...")
    bgm = joblib.load(model_path)
else:
    bgm = BayesianGaussianMixture(
        n_components=15,
        weight_concentration_prior_type='dirichlet_process',
        weight_concentration_prior=1e-3,
        max_iter=500,
        random_state=42,
        verbose=1
    )
    bgm.fit(X)
    joblib.dump(bgm, model_path)
    print(f"Model saved locally to {model_path}")

Loading saved model...


In [5]:
weights = np.round(bgm.weights_, 4)
active_clusters = np.where(weights > 0.005)[0]

print(f"active clusters: {weights[active_clusters]}")


active clusters: [0.1173 0.0324 0.0513 0.0137 0.3501 0.0557 0.1214 0.007  0.0071 0.0532
 0.0163 0.0096 0.0803 0.045  0.0396]


In [6]:
results = pd.DataFrame()
results['Cluster'] = bgm.predict(X)
results['True_Class'] = y.values 

crosstab = pd.crosstab(results['Cluster'], results['True_Class'])

crosstab = crosstab.rename(columns={0: 'Normal', 1: 'Fraud'})

if 'Fraud' not in crosstab.columns: crosstab['Fraud'] = 0
if 'Normal' not in crosstab.columns: crosstab['Normal'] = 0

crosstab['Fraud_Rate (%)'] = (crosstab['Fraud'] / (crosstab['Normal'] + crosstab['Fraud'])) * 100

print(crosstab.sort_values(by='Fraud_Rate (%)', ascending=False))

True_Class  Normal  Fraud  Fraud_Rate (%)
Cluster                                  
8             1675    346       17.120238
7             1942     61        3.045432
3             3856      7        0.181206
11            2718      4        0.146951
2            14592     10        0.068484
10            4619      3        0.064907
4            99669     39        0.039114
0            33313     11        0.033009
9            15165      5        0.032960
12           22927      4        0.017444
13           12828      1        0.007795
5            15851      1        0.006308
1             9232      0        0.000000
6            34648      0        0.000000
14           11280      0        0.000000


## XGBoosting

We isolate 2 best clusters from GMM and train an XGB on them.

In [7]:
#XGB on 2 best clusters
best_two_clusters = results[results['Cluster'].isin([7, 8])]
X_best_two_clusters = X.iloc[best_two_clusters.index]
y_best_two_clusters = y.iloc[best_two_clusters.index]

X_train, X_test, y_train, y_test = train_test_split(X_best_two_clusters, y_best_two_clusters, test_size=0.2, random_state=42, stratify=y_best_two_clusters)

neg_class_count = (y_train == 0).sum()
pos_class_count = (y_train == 1).sum()
imbalance_ratio = neg_class_count / pos_class_count

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=imbalance_ratio,
    max_depth=4,
    learning_rate=0.05,
    n_estimators=300,
    objective='binary:logistic',
    random_state=42,                
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Evaluation
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred)
print(f"True Negatives: {cm[0][0]}  |  False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}    |  True Positives: {cm[1][1]}")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

auprc = average_precision_score(y_test, y_pred_proba)
print(f"Area Under the Precision-Recall Curve (AUPRC): {auprc:.4f}")



--- Confusion Matrix ---
True Negatives: 716  |  False Positives: 8
False Negatives: 0    |  True Positives: 81

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       724
           1       0.91      1.00      0.95        81

    accuracy                           0.99       805
   macro avg       0.96      0.99      0.97       805
weighted avg       0.99      0.99      0.99       805

Area Under the Precision-Recall Curve (AUPRC): 0.9809


We find that the results are more than satisfactory as we have no false negatives on the test set. We have strong preference on model being oversensitive to being overcausious as not missing fraud is a priority.

We test the XGB trained on 2 best clusters on predicting the whole dataset

In [8]:
y_pred_whole =xgb_model.predict(X)
y_pred_proba_whole = xgb_model.predict_proba(X)[:, 1]

# Evaluation
print("\n--- Confusion Matrix ---")
cm_whole = confusion_matrix(y, y_pred_whole)
print(f"True Negatives: {cm_whole[0][0]}  |  False Positives: {cm_whole[0][1]}")
print(f"False Negatives: {cm_whole[1][0]}    |  True Positives: {cm_whole[1][1]}")

print("\n--- Classification Report ---")
print(classification_report(y, y_pred_whole))

auprc = average_precision_score(y, y_pred_proba_whole)
print(f"Area Under the Precision-Recall Curve (AUPRC): {auprc:.4f}")



--- Confusion Matrix ---
True Negatives: 284285  |  False Positives: 30
False Negatives: 83    |  True Positives: 409

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    284315
           1       0.93      0.83      0.88       492

    accuracy                           1.00    284807
   macro avg       0.97      0.92      0.94    284807
weighted avg       1.00      1.00      1.00    284807

Area Under the Precision-Recall Curve (AUPRC): 0.8486


We test the model trained on 2 best clusters on predicting the rest of the dataset

In [9]:
X_worst = X[~X.index.isin(best_two_clusters.index)] 
y_worst = y[~y.index.isin(best_two_clusters.index)]

y_pred_worst = xgb_model.predict(X_worst)
y_pred_proba_worst = xgb_model.predict_proba(X_worst)[:, 1]

# Evaluation
print("\n--- Confusion Matrix ---")
cm_worst = confusion_matrix(y_worst, y_pred_worst)
print(f"True Negatives: {cm_worst[0][0]}  |  False Positives: {cm_worst[0][1]}")
print(f"False Negatives: {cm_worst[1][0]}    |  True Positives: {cm_worst[1][1]}")

print("\n--- Classification Report ---")
print(classification_report(y_worst, y_pred_worst))

auprc = average_precision_score(y_worst, y_pred_proba_worst)
print(f"Area Under the Precision-Recall Curve (AUPRC): {auprc:.4f}")


--- Confusion Matrix ---
True Negatives: 280676  |  False Positives: 22
False Negatives: 83    |  True Positives: 2

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    280698
           1       0.08      0.02      0.04        85

    accuracy                           1.00    280783
   macro avg       0.54      0.51      0.52    280783
weighted avg       1.00      1.00      1.00    280783

Area Under the Precision-Recall Curve (AUPRC): 0.0133


We clearly see that it is, expectedly, very shit at it

We train an XGB on the whole dataset

In [10]:
X_train_whole, X_test_whole, y_train_whole, y_test_whole = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

neg_class_count_whole = (y_train_whole == 0).sum()
pos_class_count_whole = (y_train_whole == 1).sum()
imbalance_ratio_whole = neg_class_count_whole / pos_class_count_whole

xgb_model_whole = xgb.XGBClassifier(
    scale_pos_weight=imbalance_ratio_whole,
    max_depth=4,
    learning_rate=0.05,
    n_estimators=300,
    objective='binary:logistic',
    random_state=42,                
    n_jobs=-1
)

xgb_model_whole.fit(X_train_whole, y_train_whole)

xgb_model_whole.fit(X_train_whole, y_train_whole)
y_pred_whole2 = xgb_model_whole.predict(X_test_whole)
y_pred_proba_whole2 = xgb_model_whole.predict_proba(X_test_whole)[:, 1]

# Evaluation
print("\n--- Confusion Matrix ---")
cm_whole2 = confusion_matrix(y_test_whole, y_pred_whole2)
print(f"True Negatives: {cm_whole2[0][0]}  |  False Positives: {cm_whole2[0][1]}")
print(f"False Negatives: {cm_whole2[1][0]}    |  True Positives: {cm_whole2[1][1]}")

print("\n--- Classification Report ---")
print(classification_report(y_test_whole, y_pred_whole2))

auprc = average_precision_score(y_test_whole, y_pred_proba_whole2)
print(f"Area Under the Precision-Recall Curve (AUPRC): {auprc:.4f}")


--- Confusion Matrix ---
True Negatives: 56772  |  False Positives: 92
False Negatives: 12    |  True Positives: 86

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.48      0.88      0.62        98

    accuracy                           1.00     56962
   macro avg       0.74      0.94      0.81     56962
weighted avg       1.00      1.00      1.00     56962

Area Under the Precision-Recall Curve (AUPRC): 0.8477


We note that reults are almost identical as the XGB trained on 2 best clusters

We check how the big XGB predicts only 2 best clusters

In [11]:
y_pred_whole_best =xgb_model_whole.predict(X_test)
y_pred_proba_whole_best = xgb_model_whole.predict_proba(X_test)[:, 1]

# Evaluation
print("\n--- Confusion Matrix ---")
cm_whole_best = confusion_matrix(y_test, y_pred_whole_best)
print(f"True Negatives: {cm_whole_best[0][0]}  |  False Positives: {cm_whole_best[0][1]}")
print(f"False Negatives: {cm_whole_best[1][0]}    |  True Positives: {cm_whole_best[1][1]}")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred_whole_best))

auprc = average_precision_score(y_test, y_pred_proba_whole_best)
print(f"Area Under the Precision-Recall Curve (AUPRC): {auprc:.4f}")


--- Confusion Matrix ---
True Negatives: 707  |  False Positives: 17
False Negatives: 0    |  True Positives: 81

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.98      0.99       724
           1       0.83      1.00      0.91        81

    accuracy                           0.98       805
   macro avg       0.91      0.99      0.95       805
weighted avg       0.98      0.98      0.98       805

Area Under the Precision-Recall Curve (AUPRC): 0.9789


It is basically as good at it as the XGB trained only on 2 best clusters

We train XGB on the dataset without 2 best clusters

In [12]:
X_train_worst, X_test_worst, y_train_worst, y_test_worst = train_test_split(X_worst, y_worst, test_size=0.2, random_state=42, stratify=y_worst)

neg_class_count_worst = (y_train_worst == 0).sum()
pos_class_count_worst = (y_train_worst == 1).sum()
imbalance_ratio_worst = neg_class_count_worst / pos_class_count_worst

xgb_model_worst = xgb.XGBClassifier(
    scale_pos_weight=imbalance_ratio_worst,
    max_depth=4,
    learning_rate=0.05,
    n_estimators=300,
    objective='binary:logistic',
    random_state=42,                
    n_jobs=-1
)

xgb_model_worst.fit(X_train_worst, y_train_worst)

xgb_model_worst.fit(X_train_worst, y_train_worst)
y_pred_worst_worst = xgb_model_worst.predict(X_test_worst)
y_pred_proba_worst_worst = xgb_model_worst.predict_proba(X_test_worst)[:, 1]

# Evaluation
print("\n--- Confusion Matrix ---")
cm_worst = confusion_matrix(y_test_worst, y_pred_worst_worst)
print(f"True Negatives: {cm_worst[0][0]}  |  False Positives: {cm_worst[0][1]}")
print(f"False Negatives: {cm_worst[1][0]}    |  True Positives: {cm_worst[1][1]}")

print("\n--- Classification Report ---")
print(classification_report(y_test_worst, y_pred_worst_worst))

auprc = average_precision_score(y_test_worst, y_pred_proba_worst_worst)
print(f"Area Under the Precision-Recall Curve (AUPRC): {auprc:.4f}")



--- Confusion Matrix ---
True Negatives: 56053  |  False Positives: 87
False Negatives: 14    |  True Positives: 3

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56140
           1       0.03      0.18      0.06        17

    accuracy                           1.00     56157
   macro avg       0.52      0.59      0.53     56157
weighted avg       1.00      1.00      1.00     56157

Area Under the Precision-Recall Curve (AUPRC): 0.1193


We clearly see that XGB is shit at capturing patterns in anything apart from 2 best clusters

This points us to the conclussion that XGB is good at capturing basic "stupid" fraud and very bad at capturing more sophisticated, camuflaged frud.